In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

lmsys_chatbot_arena_path = kagglehub.competition_download('lmsys-chatbot-arena')
cdeotte_gemma2_9b_it_fp16_path = kagglehub.dataset_download('cdeotte/gemma2-9b-it-fp16')
linjingtingp76131220_16th_combine_19th_v1_path = kagglehub.dataset_download('linjingtingp76131220/16th-combine-19th-v1')
cdeotte_16th_place_train_1_of_3_path = kagglehub.notebook_output_download('cdeotte/16th-place-train-1-of-3')

print('Data source import complete.')


# 16th Place - Almost Gold Medal
This single model achieves `CV 0.878 and LB 0.888` in Kaggle's `LMSYS - Chatbot Arena Human Preference Predictions` competition original public LB. In the final days of the competition, a leak was discovered and the public and private test data changed.

To reproduce LB 0.888, we need to use the following settings below. And we need to run the 3 notebooks in order (Train, Quantize, Infer):

    USE_QLORA = False
    TRAIN_100_PERCENT = True
    ADD_33K = True
    DEBUG = False
    
When training without QLoRA, we cannot do this in Kaggle's 2xT4 GPU notebook, we need to use more GPU offline. I have verified that the settings above will work on 8xV100 32GB GPU or 4xA100 80GB GPU. It can probably work with less but I have not tried.

In this notebook, I train with QLoRA and quantize to 4bit in the next notebook. With these settings, we can run on Kaggle's 2xT4 GPU. If instead we train offline with LoRA and quantize to 8bit then we increase inference speed by 10% and allow us to infer max tokens = 3072, thus boosting another LB `+0.001`. Using QLoRA and 4bit (as in this notebook), we can infer 2048 and achieve `LB 0.889`

This is the second of three notebooks. Discussion about my full solution is [here][2]
* Train (Notebook 1 of 3) [here][3]
* Quantize (loads output from Train notebook)
* Infer (loads output from Quantize notebook) [here][4]

These notebooks are based on @emiz6413 great notebooks [here][1].

[1]: https://www.kaggle.com/code/emiz6413/training-gemma-2-9b-4-bit-qlora-fine-tuning
[2]: https://www.kaggle.com/competitions/lmsys-chatbot-arena/discussion/527596
[3]: https://www.kaggle.com/code/cdeotte/16th-place-train-1-of-3
[4]: https://www.kaggle.com/code/cdeotte/16th-place-infer-3-of-3

# Load Libraries

In [ ]:
!pip install -U "transformers>=4.42.3" bitsandbytes accelerate peft

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

VER=1

# FINAL SOLUTION IS USE_QLORA=FALSE, TRAIN_100_PERCENT=TRUE, ADD_33K=TRUE, DEBUG=FALSE
USE_QLORA = True
TRAIN_100_PERCENT = False
ADD_33K = False
DEBUG = True

In [ ]:
import os
import copy
from dataclasses import dataclass

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    BitsAndBytesConfig,
    Gemma2ForSequenceClassification,
    GemmaTokenizerFast,
    Gemma2Config,
    PreTrainedTokenizerBase,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import log_loss, accuracy_score

/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-12-05 15:17:38.276090: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-05 15:17:38.276146: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-05 15:17:38.277808: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to regis

# Tokenizer and Custom 3-Head Model
From EDA we see there are 58 unique models in train data after renaming some. So we hardcode the value of 58 in code below.

In [ ]:
tokenizer = GemmaTokenizerFast.from_pretrained("/kaggle/input/gemma2-9b-it-fp16")
tokenizer.add_eos_token = True  # We'll add <eos> at the end
tokenizer.padding_side = "right"

In [ ]:
import torch
import torch.nn as nn
from transformers import Gemma2ForSequenceClassification, Gemma2Config

class CustomGemma2ForSequenceClassification(Gemma2ForSequenceClassification):
    def __init__(self, config, num_labels_head1=58, num_labels_head2=58):
        super().__init__(config)
        self.num_labels_head1 = num_labels_head1
        self.num_labels_head2 = num_labels_head2
        self.classifier_head1 = nn.Linear(config.hidden_size, num_labels_head1, bias=False)
        self.classifier_head2 = nn.Linear(config.hidden_size, num_labels_head2, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        device = input_ids.device

        if labels is not None:
            labels = labels.to(device)
            outputs = super().forward(input_ids, attention_mask=attention_mask, labels=labels[:, 0], output_hidden_states=True)
        else:
            outputs = super().forward(input_ids, attention_mask=attention_mask)

        last_token_indices = (torch.sum(attention_mask, dim=1) - 1).to(device)
        last_token_outputs = outputs.hidden_states[-1].to(device)[
            torch.arange(outputs.hidden_states[-1].shape[0], device=device), last_token_indices]

        outputs_head1 = self.classifier_head1(last_token_outputs).to(device)
        outputs_head2 = self.classifier_head2(last_token_outputs).to(device)

        if labels is not None:
            labels_head1 = labels[:, 1].to(device)
            labels_head2 = labels[:, 2].to(device)

            loss_head1 = nn.CrossEntropyLoss()(outputs_head1, labels_head1)
            loss_head2 = nn.CrossEntropyLoss()(outputs_head2, labels_head2)
            loss = outputs.loss.to(device) + 0.1 * loss_head1 + 0.1 * loss_head2
            return {"loss": loss, "logits": (outputs.logits, outputs_head1, outputs_head2)}
        else:
            return {"logits": (outputs.logits, outputs_head1, outputs_head2)}

# Load Gemma2 FP16 Without Quantization

In [ ]:
from peft import PeftModel

class CustomClassificationHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.hidden_size * 2, bias=False)
        self.fc2 = nn.Linear(config.hidden_size * 2, config.hidden_size // 2, bias=False)
        self.fc3 = nn.Linear(config.hidden_size // 2, config.num_labels, bias=False)

    def forward(self, features):
        x = self.fc1(features)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

config2 = Gemma2Config.from_pretrained("/kaggle/input/gemma2-9b-it-fp16")
config2.num_labels = 3
model = CustomGemma2ForSequenceClassification.from_pretrained(
    "/kaggle/input/gemma2-9b-it-fp16",
    config=config2,
    num_labels_head1=58,
    num_labels_head2=58,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.score = CustomClassificationHead(model.config).to(next(model.parameters()).device)
model.config.use_cache = False
model = PeftModel.from_pretrained(model, "/kaggle/input/16th-combine-19th-v1")
model

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of CustomGemma2ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/gemma2-9b-it-fp16 and are newly initialized: ['classifier_head1.weight', 'classifier_head2.weight', 'score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): CustomGemma2ForSequenceClassification(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict(

# Merge Adapter to Gemma2 FP16

In [ ]:
model = model.merge_and_unload()
model

CustomGemma2ForSequenceClassification(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3584, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=3584, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (pre_feedforward_

# Save Merged Model

In [ ]:
model.save_pretrained(f"merged-v{VER}")
tokenizer.save_pretrained(f"merged-v{VER}")

('merged-v1/tokenizer_config.json',
 'merged-v1/special_tokens_map.json',
 'merged-v1/chat_template.jinja',
 'merged-v1/tokenizer.model',
 'merged-v1/added_tokens.json',
 'merged-v1/tokenizer.json')

In [ ]:
del model
torch.cuda.empty_cache()

# Quantize to 8bit (or 4bit)
We will load the merged model from disk, quantize it, and save quantized model to disk.

In [ ]:
from transformers import BitsAndBytesConfig
if not USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_8bit = True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules = ["score","classifier_head1", "classifier_head2"]
    )
else: # USE SAME QUANTIZATION THAT WE TRAINED WITH
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4", #nf4 or fp4
        bnb_4bit_use_double_quant = False,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules = ["score","classifier_head1", "classifier_head2"]
    )

In [ ]:
config2 = Gemma2Config.from_pretrained("/kaggle/input/gemma2-9b-it-fp16")
config2.num_labels = 3
model = CustomGemma2ForSequenceClassification.from_pretrained(
    f"merged-v{VER}",
    config=config2,
    num_labels_head1=58,
    num_labels_head2=58,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config = bnb_config,
)
model.config.use_cache = False
model

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of CustomGemma2ForSequenceClassification were not initialized from the model checkpoint at merged-v1 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CustomGemma2ForSequenceClassification(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=3584, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=3584, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((3584,), eps=1e-0

# Save Quantized Model

In [ ]:
os.system(f"rm -r merged-v{VER}")

0

In [ ]:
if USE_QLORA:
    model.save_pretrained(f"16th_combine_19th_merged-v{VER}-4bit")
    tokenizer.save_pretrained(f"16th_combine_19th_merged-v{VER}-4bit")
else:
    model.save_pretrained(f"16th_combine_19th_merged-v{VER}-8bit")
    tokenizer.save_pretrained(f"16th_combine_19th_merged-v{VER}-8bit")